# 05 Machine Learning Models

Train tabular regression models for temperature prediction, compare results, and save the best model.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "GlobalWeatherRepository.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from src.features import prepare_model_frame
from src.preprocessing import load_processed_data
from src.train import compare_regression_models, save_model
from src.visualize import save_figure


In [3]:
anomaly_free_path = PROCESSED_DIR / "weather_without_anomalies.csv"
clean_path = PROCESSED_DIR / "clean_weather_data.csv"
source_path = anomaly_free_path if anomaly_free_path.exists() else clean_path

df = load_processed_data(source_path)
X, y = prepare_model_frame(df, target_column="temperature_celsius")

print(f"Loaded source data from: {source_path}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
display(X.head())


Loaded source data from: d:\Projects 2025\weather-forecasting-eda-ml\data\processed\weather_without_anomalies.csv
Feature matrix shape: (130542, 42)
Target shape: (130542,)


,latitude,longitude,last_updated_epoch,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,precip_mm,...,month,day,day_of_week,day_of_year,hour,month_sin,month_cos,hour_sin,hour_cos,daylight_hours
0,46.60,-120.49,1715849100,61.0,4.3,6.8,220,1012.0,29.87,0.00,...,5,16,3,137,1,0.5,-0.866025,0.258819,0.965926,15.083333
1,14.10,-87.22,1715849100,73.4,3.8,6.1,240,1017.0,30.03,0.28,...,5,16,3,137,2,0.5,-0.866025,0.500000,0.866025,12.783333
2,13.71,-89.20,1715849100,78.8,2.2,3.6,182,1010.0,29.81,0.30,...,5,16,3,137,2,0.5,-0.866025,0.500000,0.866025,12.766667
3,17.25,-88.77,1715849100,78.9,4.3,6.8,99,1007.0,29.75,0.00,...,5,16,3,137,2,0.5,-0.866025,0.500000,0.866025,12.950000
4,12.15,-86.27,1715849100,80.9,3.6,5.8,120,1009.0,29.80,0.01,...,5,16,3,137,2,0.5,-0.866025,0.500000,0.866025,12.683333


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

models, results_df = compare_regression_models(X_train, X_test, y_train, y_test)
display(results_df)


,mae,rmse,mape,r2
linear_regression,0.018449,0.022676,0.191699,0.999994
random_forest,0.006685,0.192971,0.051858,0.999555
gradient_boosting,0.048787,0.201980,0.546000,0.999513


In [5]:
best_model_name = results_df.index[0]
best_model = models[best_model_name]
best_model_path = MODELS_DIR / f"{best_model_name}_temperature_model.joblib"
save_model(best_model, best_model_path)

metrics_path = PROJECT_ROOT / "reports" / "temperature_model_metrics.csv"
results_df.to_csv(metrics_path)

print(f"Best model: {best_model_name}")
print(f"Saved best model to: {best_model_path}")
print(f"Saved comparison metrics to: {metrics_path}")


Best model: linear_regression
Saved best model to: d:\Projects 2025\weather-forecasting-eda-ml\models\linear_regression_temperature_model.joblib
Saved comparison metrics to: d:\Projects 2025\weather-forecasting-eda-ml\reports\temperature_model_metrics.csv


In [6]:
if hasattr(best_model, "feature_importances_"):
    feature_importance = (
        pd.Series(best_model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    feature_importance.plot(kind="bar", ax=ax, color="teal")
    ax.set_title(f"Top Feature Importances ({best_model_name})")
    ax.set_ylabel("Importance")
    plt.tight_layout()
    display(fig)
    save_figure(fig, FIGURES_DIR / "ml_feature_importance.png")
    plt.close(fig)
else:
    print(f"{best_model_name} does not expose feature_importances_.")


linear_regression does not expose feature_importances_.
